### CONTENT-BASED MOVIE RECOMMENDATION SYSTEM
### Dataset: IMDB Genres (jquigl/imdb-genres)
### Method: TF-IDF + Cosine Similarity (No pre-trained embeddings)

SECTION 1: LOAD DATASET

In [1]:
from datasets import load_dataset

dataset = load_dataset("jquigl/imdb-genres")

train_data = dataset["train"]
val_data = dataset["validation"]

train_data[0]

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'movie title - year': 'Flaming Ears - 1992',
 'genre': 'Fantasy',
 'expanded-genres': 'Fantasy, Sci-Fi',
 'rating': 6.0,
 'description': 'Flaming Ears is a pop sci-fi lesbian fantasy feature set in the year 2700 in the fictive burned-out city of Asche. It follows the tangled lives of three women - Volley, Nun and Spy.'}

SECTION 2: CONVERT TO DATAFRAME

In [2]:
import pandas as pd

df = train_data.to_pandas()
val_df = val_data.to_pandas()

print("Train data shape: ", df.shape)
df.head(3)

Train data shape:  (238256, 5)


,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."


SECTION 3: EXPLORATORY DATA ANALYSIS (TRAIN SET)

In [6]:
# Check for null values, duplicates and NaN values in rating
print("Null counts:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("String NaN in rating:", (df['rating'] == 'NaN').sum())

Null counts:
 movie title - year        0
genre                     0
expanded-genres           0
rating                69721
description               0
dtype: int64

Duplicates: 18
String NaN in rating: 0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238256 entries, 0 to 238255
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movie title - year  238256 non-null  object 
 1   genre               238256 non-null  object 
 2   expanded-genres     238256 non-null  object 
 3   rating              168535 non-null  float64
 4   description         238256 non-null  object 
dtypes: float64(1), object(4)
memory usage: 9.1+ MB


In [50]:
import pandas as pd

df = train_data.to_pandas()
df.head()

,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."
3,Gulliver Returns - 2021,Fantasy,"Animation, Adventure, Family",4.4,The legendary Gulliver returns to the Kingdom ...
4,Prithvi Vallabh - 1924,Biography,"Biography, Drama, Romance",NaN,"Seminal silent historical film, the story feat..."


In [51]:
val_df = val_data.to_pandas()
val_df.head(3)

,movie title - year,genre,expanded-genres,rating,description
0,The Dictator - 1922,Adventure,"Adventure, Comedy, Drama",2.6,"Pursued by a cab driver for an unpaid fare, a ..."
1,EDSA XXX: Nothing Ever Changes in the Ever-Cha...,Fantasy,"Drama, Fantasy, Musical",4.6,Political realism in an absurdist musical. Eve...
2,Flaming Waters - 1925,Action,"Action, Drama",7.0,"After several years' absence, the young sailor..."


In [4]:
int(df["rating"].isnull().sum())

69721

In [5]:
df.shape

(238256, 5)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238256 entries, 0 to 238255
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movie title - year  238256 non-null  object 
 1   genre               238256 non-null  object 
 2   expanded-genres     238256 non-null  object 
 3   rating              168535 non-null  float64
 4   description         238256 non-null  object 
dtypes: float64(1), object(4)
memory usage: 9.1+ MB


In [7]:
(df['rating']=='NaN').sum()

np.int64(0)

In [8]:
df.isnull().sum()

movie title - year        0
genre                     0
expanded-genres           0
rating                69721
description               0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(18)

In [10]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 238238 entries, 0 to 238255
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movie title - year  238238 non-null  object 
 1   genre               238238 non-null  object 
 2   expanded-genres     238238 non-null  object 
 3   rating              168533 non-null  float64
 4   description         238238 non-null  object 
dtypes: float64(1), object(4)
memory usage: 10.9+ MB


In [12]:
df.isnull().sum()

movie title - year        0
genre                     0
expanded-genres           0
rating                69705
description               0
dtype: int64

In [13]:
df.head(3)

,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."


In [14]:
df['title'] = df['movie title - year'].str.split(' -').str[0]
df['year'] = df['movie title - year'].str.split(' -').str[1]

In [15]:
df['tags'] = df['description'] + df['genre'] + df['expanded-genres']
df.head(1)

,movie title - year,genre,expanded-genres,rating,description,title,year,tags
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...,Flaming Ears,1992,Flaming Ears is a pop sci-fi lesbian fantasy f...


In [16]:
movies = df[['title','tags', 'rating', 'year' ]]
movies.head(2)

,title,tags,rating,year
0,Flaming Ears,Flaming Ears is a pop sci-fi lesbian fantasy f...,6.0,1992
1,Jeg elsker dig,Six people - three couples - meet at random at...,5.8,1957


In [17]:
movies['tags'][1]

'Six people - three couples - meet at random at a dance restaurant in the Copenhagen nightlife. A marriage swindler and former actress, an elderly Supreme Court attorney with wife as well as...                See full summary\xa0»RomanceComedy, Drama, Romance'

In [18]:
# movies['tags'] = movies['tags'].apply(lambda x:x.lower())
# movies['tags'] = movies['tags'].fillna("").str.lower()
movies.loc[:, 'tags'] = movies['tags'].fillna("").str.lower()

In [19]:
movies.loc[:, 'tags'] = movies['tags'].str.replace(r'[^\w\s]', '', regex=True)

In [20]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

movies = movies.copy()

movies['tags'] = (
    movies['tags']
    .fillna("")
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)   # remove punctuation
    .str.replace(r'\s+', ' ', regex=True)      # remove extra spaces
    .apply(lambda x: " ".join([w for w in x.split() if w not in stop_words]))
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Data Preprocessing

In [21]:
# from sklearn.feature_extraction.text import CountVectorizer

# cv = CountVectorizer(
#     max_features=20000,
#     min_df=5,
#     max_df=0.8
# )

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(
    max_features=15000,
    min_df=5,
    stop_words='english'
)

In [23]:
vectors = tf.fit_transform(movies['tags'])

In [24]:
vectors.shape

(238238, 15000)

In [25]:
tf.get_feature_names_out()

array(['000', '10', '100', ..., 'zorro', 'zoya', 'zurich'],
      shape=(15000,), dtype=object)

Stemming

In [26]:
import nltk

In [27]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [28]:
def stem(text):
    y = []
    
    for i in text.split():
        ps.stem(i)        
    
    return " ".join(y)

In [29]:
# movies['tags'].apply(stem)

In [30]:
# tf.get_feature_names_out()[1000:1500]

In [31]:
movies.head(3)

,title,tags,rating,year
0,Flaming Ears,flaming ears pop scifi lesbian fantasy feature...,6.0,1992
1,Jeg elsker dig,six people three couples meet random dance res...,5.8,1957
2,Povjerenje,small unnamed town year 2025 krsto works agenc...,NaN,2021


Cosine Similarity

In [32]:
# Start from 0
movies['movie_id'] = range(len(movies))

# OR start from 1
# movies['movie_id'] = range(1, len(movies)+1)

In [33]:
movies[['movie_id', 'title', 'tags']].head(3)

,movie_id,title,tags
0,0,Flaming Ears,flaming ears pop scifi lesbian fantasy feature...
1,1,Jeg elsker dig,six people three couples meet random dance res...
2,2,Povjerenje,small unnamed town year 2025 krsto works agenc...


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_movies(input_text, top_k=5):
    """
    Recommend top_k movies similar to input_text.
    input_text: movie title or any keywords
    Returns: list of movie titles
    """
    input_text_lower = input_text.lower()
    
    # 1️⃣ Try to find exact title match
    matched = movies[movies['title'].str.lower() == input_text_lower]
    
    if not matched.empty:
        # Exact title found → use its vector
        movie_idx = matched.index[0]
        input_vector = vectors[movie_idx]
        ignore_idx = {movie_idx}
    else:
        # Treat input as keywords → convert to TF-IDF
        input_vector = tf.transform([input_text])
        ignore_idx = set()
    
    # 2️⃣ Compute cosine similarity with all movie vectors
    sim_scores = cosine_similarity(input_vector, vectors).flatten()
    
    # 3️⃣ Get top indices sorted by similarity
    sorted_indices = np.argsort(-sim_scores)
    
    # 4️⃣ Filter out ignored indices & duplicates efficiently
    recommended = []
    seen_titles = set()
    for idx in sorted_indices:
        title = movies.iloc[idx]['title']
        if idx not in ignore_idx and title not in seen_titles:
            recommended.append(title)
            seen_titles.add(title)
        if len(recommended) >= top_k:
            break
    
    return recommended

In [46]:
recommend_movies('romance')

['Once Upon a Time in the Midwest',
 'En flicka för mej',
 'The Secret Life of Walter Mitty',
 'A Stranger in Town',
 'Okey, Mister Pancho']

Preprocess validation set

In [53]:
val_df = val_data.to_pandas()
val_df = val_df.drop_duplicates()
val_df.sample(3)

,movie title - year,genre,expanded-genres,rating,description
25610,Virtual Reality - nan,Action,Action,NaN,A woman with no memory of her past life finds ...
27078,O Último Condenado à Morte - 2009,Biography,"Biography, Drama, History",6.3,The story about the last man to be sentenced t...
9163,Abduction - I,Fantasy,Fantasy,NaN,A couple wake to find themselves locked in a r...


In [55]:
# Get title and year
val_df['title'] = val_df['movie title - year'].str.split(' -').str[0]
val_df['year']  = val_df['movie title - year'].str.split(' -').str[1]
val_df.sample(3)

,movie title - year,genre,expanded-genres,rating,description,title,year
11305,Una ombra en el jardí - 1989,Thriller,Thriller,7.8,"Luis, a botanical researcher, is suddenly aban...",Una ombra en el jardí,1989
5806,Metal Messiah - 1978,Scifi,"Musical, Sci-Fi",5.9,A dystopian rock opera about an enigmatic meta...,Metal Messiah,1978
26146,A Demon in My View - 2013,Horror,"Horror, Thriller",3.9,"Leah, a freshman at Nolan University, is strug...",A Demon in My View,2013


In [56]:
val_df['tags'] = val_df['description'] + val_df['genre'] + val_df['expanded-genres']


In [57]:
# data preprocessing like in train data 
val_df['tags'] = (
    val_df['tags']
    .fillna("")
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .apply(lambda x: " ".join(
        [w for w in x.split() if w not in stop_words]
    ))
    .apply(stem)
)

In [58]:
val_movies = val_df[['title', 'tags', 'rating', 'year', 'genre']].copy()
val_movies['movie_id'] = range(len(val_movies))

print("Validation set shape:", val_movies.shape)
val_movies.head(3)

Validation set shape: (29808, 6)


,title,tags,rating,year,genre,movie_id
0,The Dictator,,2.6,1922,Adventure,0
1,EDSA XXX: Nothing Ever Changes in the Ever-Cha...,,4.6,2014,Fantasy,1
2,Flaming Waters,,7.0,1925,Action,2


Manual testing

In [59]:
# =============================================================================
# SECTION 11: TESTING — MANUAL RECOMMENDATION QUERIES
# =============================================================================

print("\n" + "="*50)
print("RECOMMENDATION TESTS")
print("="*50)

# Test 1: Free-text genre keyword
print("\n[Test 1] Query: 'romance'")
print(recommend_movies('romance'))

# Test 2: Descriptive keywords
print("\n[Test 2] Query: 'action thriller spy'")
print(recommend_movies('action thriller spy'))

# Test 3: Exact movie title (will use its own vector)
print("\n[Test 3] Query: exact movie title from train set")
sample_title = movies['title'].iloc[0]
print(f"  Title: '{sample_title}'")
print(recommend_movies(sample_title))

# Test 4: A movie from the validation set (unseen during training)
print("\n[Test 4] Query: movie title from validation set")
val_sample_title = val_movies['title'].iloc[0]
print(f"  Title: '{val_sample_title}'")
print(recommend_movies(val_sample_title))

# Test 5: Description-style query
print("\n[Test 5] Query: 'a young wizard discovers magical powers'")
print(recommend_movies('a young wizard discovers magical powers'))



RECOMMENDATION TESTS

[Test 1] Query: 'romance'
['Once Upon a Time in the Midwest', 'En flicka för mej', 'The Secret Life of Walter Mitty', 'A Stranger in Town', 'Okey, Mister Pancho']

[Test 2] Query: 'action thriller spy'
['Untitled Idris Elba/Simon Kinberg/Audrey Chon Africa-Set Spy Romance Project', 'Secret Female Investigator: Wager on Lips', 'Are You Ready?', 'Sanctuary', 'Violinist in Belgium']

[Test 3] Query: exact movie title from train set
  Title: 'Flaming Ears'
['Flaming Ears', 'Elliptica', 'Love(less)', 'Madame X: An Absolute Ruler', 'Mad Mountain']

[Test 4] Query: movie title from validation set
  Title: 'The Dictator'
['Octonauts & the Great Barrier Reef', 'Nathan and the Luthier', 'Lords of the Deep', 'Beneath the 12-Mile Reef', 'Stratosphere']

[Test 5] Query: 'a young wizard discovers magical powers'
['Gekijouban Mahou sensei Negima! Anime Final', 'Aggi Meeda Guggilam', 'Syupeo Hong Gil-dong 3', "Fire Dragon's Breath", 'Sade']
